In [3]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import glob
from matplotlib import colormaps
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex

In [4]:
load_dotenv(override=True)

MODEL = "gpt-4.1-nano"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path.cwd().parent.parent / "outputs/clean"
AVERAGE_CHUNK_SIZE = 2000

openai = OpenAI()

In [5]:
# Similar to LangChain's Document

class Result(BaseModel):
    page_content: str
    metadata: dict

In [6]:
class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions, including the topic and category of this chunk")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text, metadata=metadata)

class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:
1. Fetch documents from the kb
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

In [7]:
def fetch_documents():
    """Similar to LangChain DirectoryLoader"""

    documents = []
    filenames = glob.glob(str(KNOWLEDGE_BASE_PATH)+"/*.md")

    for filename in filenames:
        doc_type = Path(filename).stem
        with open(filename, "r", encoding="utf-8") as f:
            documents.append({"type": doc_type, "source": filename, "text": f.read()})

    print(f"Loaded {len(documents)} documents")

    return documents

In [8]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the public webpages of a company called SW Waiblingen.
The document from the category: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [9]:
documents = fetch_documents()
print(make_prompt(documents[0]))

Loaded 81 documents

You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the public webpages of a company called SW Waiblingen.
The document from the category: Geschäftskunden_Strom_Grundversorgung
The document has been retrieved from: /Users/sunzeyuan/projects/agent_exercises/crawler copy/outputs/clean/Geschäftskunden_Strom_Grundversorgung.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 1 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Togeth

In [10]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [11]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the public webpages of a company called SW Waiblingen.\nThe document from the category: Geschäftskunden_Strom_Grundversorgung\nThe document has been retrieved from: /Users/sunzeyuan/projects/agent_exercises/crawler copy/outputs/clean/Geschäftskunden_Strom_Grundversorgung.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 1 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original t

In [12]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    try:
        doc_as_chunks = Chunks.model_validate_json(reply).chunks
    except ValidationError as e:
        print(f"Failed to parse response: {reply[:1000]}...")
        print(f"Error: {e}")
        return []
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [13]:
process_document(documents[0])

[Result(page_content="Introduction to Geschäftskunden - Strom - Grundversorgung\n\nThis section explains the basics of basic supply ('Grundversorgung') in electricity, which is a legally secured electricity supply for residential customers when no other contract exists.\n\nDie Grundversorgung ist die gesetzlich gesicherte Stromlieferung für Haushaltskunden, wenn kein anderer Stromvertrag abgeschlossen wurde – zuverlässig, transparent und jederzeit verfügbar.", metadata={'source': '/Users/sunzeyuan/projects/agent_exercises/crawler copy/outputs/clean/Geschäftskunden_Strom_Grundversorgung.md', 'type': 'Geschäftskunden_Strom_Grundversorgung'}),
 Result(page_content='Downloads related to Strom Grundversorgung\n\nThis section provides links to various downloadable documents related to electricity basic supply, including contracts, conditions, notices, and price sheets from 2023 to 2026.\n\n##  Downloads Strom Grundversorgung\n  *  Abwendungsvereinbarung Muster 2024 (PDF | 189 KB)\n  *  Ergän

/Users/sunzeyuan/projects/agent_exercises/crawler copy/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='{"chunks...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


In [14]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [15]:
chunks = create_chunks(documents)

  0%|          | 0/81 [00:00<?, ?it/s]

100%|██████████| 81/81 [06:33<00:00,  4.85s/it]


In [16]:
print(len(chunks))

255


In [17]:
chroma = PersistentClient(path=DB_NAME)
chroma.list_collections()


[Collection(name=docs)]

In [18]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")
    

In [19]:
create_embeddings(chunks)

Vectorstore created with 255 documents


In [20]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]


doc_types = [metadata['type'] for metadata in metadatas]

# Colour by section - the part before the first "_" - not by the full filename
# stem: there are 81 distinct stems but tab20 only has 20 colours, so stems
# would wrap round and four unrelated pages would share a colour.
sections = [t.split('_')[0] for t in doc_types]

# sorted(), not set(): set iteration order changes between kernel restarts, so
# the same section would get a different colour on every run.
unique_sections = sorted(set(sections))

# Generate distinct colors for all sections
cmap = plt.get_cmap('tab20')

# to_hex is the point: cmap(i) is an RGBA tuple, which plotly.js cannot read as
# a colour. Plotly accepts it without error and then falls back to its default,
# painting every marker the same blue. Hex strings are what it expects.
color_map = {
    section: to_hex(cmap(i % len(cmap.colors)))
    for i, section in enumerate(unique_sections)
}

# Apply colors
colors = [color_map[section] for section in sections]

print(f"{len(unique_sections)} sections, {len(cmap.colors)} colours available")
print(f"Color map: {color_map}")

10 sections, 20 colours available
Color map: {'Aktuelles': '#1f77b4', 'Geschäftskunden': '#aec7e8', 'Karriere': '#ff7f0e', 'Kontakt': '#ffbb78', 'Kundenportal': '#2ca02c', 'Netze': '#98df8a', 'Privatkunden': '#d62728', 'Störung': '#ff9896', 'Unternehmen': '#9467bd', 'Wissensdatenbank': '#c5b0d5'}


In [21]:
unique_sections

['Aktuelles',
 'Geschäftskunden',
 'Karriere',
 'Kontakt',
 'Kundenportal',
 'Netze',
 'Privatkunden',
 'Störung',
 'Unternehmen',
 'Wissensdatenbank']

In [22]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [23]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from msot relevant to least relevant, by chunk id number"
    )

In [24]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [38]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [39]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [41]:
for chunk in chunks:
    print(chunk.page_content[:100]+"...")

Determination as the Gas Basic Supplier in Waiblingen

SW Waiblingen GmbH was appointed as the basic...
Current and Past Designations as the Public Utility Provider

This section details the designation o...
Geschäftsführung – Verantwortliche Personen

Information about the leadership of the company, includ...
Fiber Optic Expansion in Waiblingen's Business Districts (Part 1)

The fiber optic network has been ...
Overview of Power Labeling for 2025

This section provides a summary of the electricity labeling by ...
Neue Tochtergesellschaft – Nachhaltige Energielösungen Waiblingen GmbH (NEW)

Introduction of the co...
Ableseprozess für Einspeiseanlagen

Der Ableseprozess findet einmal jährlich im Dezember statt, bei ...
Introduction to Tarifübersicht Freibäder 2026

This document provides the tariff overview for public...
Information zum intelligenten Messsystem (iMSys)

Beschreibung der Anforderungen und Funktionalitäte...
Ermäßigungen für Jugendliche und Erwachsene sowie Gültige Auswei

In [42]:
reranked = rerank(question, chunks)

[6, 5, 9, 4, 8, 7, 3, 2, 1, 10, 4, 3, 2, 1]


/Users/sunzeyuan/projects/agent_exercises/crawler copy/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='{"order"...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


In [43]:
for chunk in reranked:
    print(chunk.page_content[:50]+"...")

Neue Tochtergesellschaft – Nachhaltige Energielösu...
Overview of Power Labeling for 2025

This section ...
Information zum intelligenten Messsystem (iMSys)

...
Fiber Optic Expansion in Waiblingen's Business Dis...
Introduction to Tarifübersicht Freibäder 2026

Thi...
Ableseprozess für Einspeiseanlagen

Der Ableseproz...
Geschäftsführung – Verantwortliche Personen

Infor...
Current and Past Designations as the Public Utilit...
Determination as the Gas Basic Supplier in Waiblin...
Ermäßigungen für Jugendliche und Erwachsene sowie ...
Fiber Optic Expansion in Waiblingen's Business Dis...
Geschäftsführung – Verantwortliche Personen

Infor...
Current and Past Designations as the Public Utilit...
Determination as the Gas Basic Supplier in Waiblin...


In [44]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [45]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company SW Waiblingen.
You are chatting with a user about SW Waiblingen.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [46]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [47]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company SW Waiblingen.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [48]:
rewrite_query("Who won the IIOTY award?", [])

/Users/sunzeyuan/projects/agent_exercises/crawler copy/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='Who rece...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


'Who received the IIOTY award?'

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query) # find chunks and rerank directly
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks


In [64]:
answer_question("Who won the IIOTY award?", [])

Who was the recipient of the IIOTY award?


/Users/sunzeyuan/projects/agent_exercises/crawler copy/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='Who was ...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


[8, 4, 1, 10, 6, 9, 7, 3, 2, 5]


/Users/sunzeyuan/projects/agent_exercises/crawler copy/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content="I don't ...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


("I don't have information about the winner of the IIOTY award. For the most current details, I recommend checking the official IIOTY website or related press releases.",
 [Result(page_content="Presse- und Kommunikationseinstellungen\n\nContact information for the company's public relations and marketing team, providing channels for media inquiries and corporate communication.\n\n###  Unternehmenskommunikation & Marketing +49 7151 131-198 E-Mail senden", metadata={'source': '/Users/sunzeyuan/projects/agent_exercises/crawler copy/outputs/clean/Unternehmen.md', 'type': 'Unternehmen'}),
  Result(page_content='Vertragspartner für Glasfaser-Dienste: NetCom BW\n\nNetCom BW is the official partner providing internet and telephony services to businesses in Waiblingen, handling contracts and inquiries related to fiber optic connectivity.\n\nDer Vertragspartner der Stadtwerke, die NetCom BW, bietet für die ansässigen Gewerbetreibenden leistungsstarke Dienste für Internet und Telefonie an. Bei al

In [55]:
chunks = fetch_context(query)

[10, 9, 8, 7, 6, 5, 4, 3, 2, 1]


/Users/sunzeyuan/projects/agent_exercises/crawler copy/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='{"order"...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


In [57]:
messages = make_rag_messages(question, [], chunks)

In [61]:
response = completion(model=MODEL, messages=messages)
#messages

/Users/sunzeyuan/projects/agent_exercises/crawler copy/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content="I'm sorr...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
